In [1]:
import xarray as xr
import pandas as pd
from pathlib import Path
import os
import cdsapi
from netCDF4 import Dataset
import matplotlib.pyplot as plt
import geopandas as gpd
import numpy as np
from shapely.geometry import Point, Polygon
from pyproj import Transformer

# Beetle data

## SLU

In [2]:
t = Transformer.from_crs(3006, 4326, always_xy=True)

In [3]:
directory = os.fsencode('./data/beetle/slu')
    
df_merged = pd.DataFrame(columns=['Plotno', 'Season', 'Date', 'Plot_Area', 'Proportion_of_Norway_spruce',
       'GPS-Nord_Scrambled', 'GPS-Ost_Scrambled'])

b_list = []

cols = ['Plotno', 'Date', 'Plot_Area', 'Proportion_of_Norway_spruce',
       'GPS-Nord_Scrambled', 'GPS-Ost_Scrambled']

for year in ['2011','2012','2013','2014','2015','2016','2020']:
    plot = pd.read_csv(f'./data/beetle/slu/{year}_yta.csv', sep=";", skiprows=1,  on_bad_lines='skip', encoding='latin1')
    plot = plot[['Plotno','Date', 'Plot_Area', 'Proportion_of_Norway_spruce', 'GPS-Nord_Scrambled', 'GPS-Ost_Scrambled']]

    trees = pd.read_csv(f'./data/beetle/slu/{year}_trad.csv', sep=";", skiprows=1)
    trees = trees[['Plotno','Season']]

    merge = trees.merge(plot, on='Plotno', how='left')
    b_list.append(merge)

df_merged = pd.concat(b_list, ignore_index=True)
df_merged.to_csv('./data/beetle/lsu_processed.csv', sep=',')
df_merged['Date'] = df_merged['Date'].astype(str).str.replace('.0', '', regex=False)
df_merged['Date'] = pd.to_datetime(df_merged['Date'], errors='coerce')

# Transformer: SWEREF99 TM (EPSG:3006) → WGS84 (EPSG:4326)
df_merged["Lon"], df_merged["Lat"] = t.transform(
    df_merged["GPS-Ost_Scrambled"],
    df_merged["GPS-Nord_Scrambled"]
)
df_merged.drop(columns=['GPS-Nord_Scrambled', 'GPS-Ost_Scrambled'], inplace=True)
df_merged.to_csv('./data/beetle/lsu_processed.csv')


In [218]:
beetle_cv = pd.read_csv('./data/beetle/lsu_processed.csv')
beetle_cv

,Unnamed: 0,Plotno,Season,Date,Plot_Area,Proportion_of_Norway_spruce,Lon,Lat
0,0,3,0,2011-10-04,1963.0,100.0,19.212779,63.493099
1,1,4,2,2011-10-05,1963.0,100.0,18.562187,63.546142
2,2,4,1,2011-10-05,1963.0,100.0,18.562187,63.546142
3,3,4,1,2011-10-05,1963.0,100.0,18.562187,63.546142
4,4,4,1,2011-10-05,1963.0,100.0,18.562187,63.546142
...,...,...,...,...,...,...,...,...
2492,2492,867,0,2020-10-05,"1963,5",100.0,60.764318,3.441587
2493,2493,867,0,2020-10-05,"1963,5",100.0,60.764318,3.441587
2494,2494,867,0,2020-10-05,"1963,5",100.0,60.764318,3.441587
2495,2495,867,0,2020-10-05,"1963,5",100.0,60.764318,3.441587


## GBIF (old)

In [52]:
def parse_first_date(date_str):
    if pd.isna(date_str):
        return pd.NaT  # Return pandas missing datetime
    first_part = str(date_str).split('/')[0]  # handle range
    date_only = first_part.split('T')[0]      # remove time
    return date_only

In [90]:
csv = pd.read_csv('./data/beetle/gbif_raw.csv', sep='\t')
csv = csv[['eventDate', 'countryCode','stateProvince','decimalLatitude', 'decimalLongitude', 'coordinateUncertaintyInMeters']]
csv['dateTime'] = pd.to_datetime(csv['eventDate'].apply(parse_first_date), format='%Y-%m-%d', errors='coerce')
csv.drop(columns=['eventDate'], inplace=True)
csv = csv[csv['dateTime'].dt.year >= 1950]
csv.to_csv('./data/beetle/gbif_processed.csv', sep=',')
csv


/var/folders/tm/18h0f9s95nn6kwcr2df09m4w0000gn/T/ipykernel_45626/3788725964.py:1: DtypeWarning: Columns (39,45) have mixed types. Specify dtype option on import or set low_memory=False.
  csv = pd.read_csv('./data/beetle/gbif_raw.csv', sep='\t')


,countryCode,stateProvince,decimalLatitude,decimalLongitude,coordinateUncertaintyInMeters,dateTime
0,SE,Medelpad,62.49180,17.53122,75.0,2014-07-31
1,SE,Jämtland,62.70294,15.63134,50.0,2014-07-01
2,SE,Västmanland,59.45682,14.89153,50.0,2014-06-10
3,SE,Södermanland,59.12609,16.35427,500.0,2002-05-01
4,SE,Södermanland,59.21937,16.94976,150.0,2002-05-01
...,...,...,...,...,...,...
17340,SE,Öland,57.26623,17.03666,100.0,1997-10-18
17341,SE,Hälsingland,61.59949,16.94856,100.0,1972-10-19
17342,SE,Södermanland,59.10724,16.79408,25.0,2014-09-17
17343,SE,Södermanland,59.44316,16.32641,25.0,2014-08-29
